# Phase 0.5 — GS-Sub greedy baseline (n=2) over MS(1190)

Pure 2-generator greedy (no `z=w`, no stabilization). Set **CAP** and **BUDGET** below for your box,
then Runtime -> Run all. Each run is **independent and resumable** (skips idx already recorded), so a
pre-empted Colab session just resumes. Results are written to your Drive folder; copy them into the
repo's `results/` when done.

| box | CAP | BUDGET | writes |
|---|---|---|---|
| 1 | `'sum'`         | `100_000`   | `greedy_reprogate_100k.jsonl` |
| 2 | `'sum'`         | `1_000_000` | `greedy_reprogate_1m.jsonl` |
| 3 | `'per_relator'` | `100_000`   | `greedy_baseline_100k.jsonl` |
| 4 | `'per_relator'` | `1_000_000` | `greedy_baseline_1m.jsonl` |

`sum` reproduces the paper's 634@100k / 640@1m; `per_relator` (L=24) is the baseline the future
`z=w` arms compare against. Run the last cell **once on any box** to make `labels_1190.json`, and
again **after all 4 streams exist** to build the minimum-path-length index.

In [ ]:
# ============ EDIT FOR YOUR BOX ============
CAP     = 'sum'          # 'sum' (reprogate) | 'per_relator' (baseline)
BUDGET  = 100_000        # 100_000 | 1_000_000
MAX_LEN = None           # None -> 100 (sum) / 24 (per_relator)
WORKERS = 4              # CPU cores to use (main process is the only JSONL writer -> safe)
SHARD   = None           # e.g. '0/3' to run only idx-chunk 0 of 3 (optional, for extra boxes)

# ============ repo / output ============
REPO_URL  = 'https://github.com/Avi161/ACSolverX.git'
BRANCH    = 'test/stable-ac-moves'
DRIVE_OUT = '/content/drive/MyDrive/acsolverx_results'  # streams persist here across preemption

In [ ]:
import os, sys, subprocess

IN_COLAB = 'google.colab' in sys.modules or os.path.isdir('/content')
REPO_DIR = '/content/ACSolverX' if IN_COLAB else os.path.abspath(os.path.join(os.getcwd()))

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR], check=True)
    subprocess.run(['pip', 'install', '-q', 'numba', 'numpy'], check=True)
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = DRIVE_OUT
else:
    # running locally in the repo: find repo root, write to results/
    while not os.path.isdir(os.path.join(REPO_DIR, 'data')) and REPO_DIR != '/':
        REPO_DIR = os.path.dirname(REPO_DIR)
    OUT_DIR = os.path.join(REPO_DIR, 'results')

BASE = os.path.join(REPO_DIR, 'experiments', 'stable_ac', 'one_generator', 'baseline_n2')
sys.path.insert(0, BASE)
os.makedirs(OUT_DIR, exist_ok=True)
DATA_PATH = os.path.join(REPO_DIR, 'data', '1190MS.txt')
print('REPO_DIR:', REPO_DIR); print('OUT_DIR :', OUT_DIR)

In [ ]:
import run_greedy

shard = None
if SHARD:
    _i, _k = SHARD.split('/'); shard = (int(_i), int(_k))
max_len = MAX_LEN if MAX_LEN is not None else run_greedy.DEFAULT_MAX_LEN[CAP]

# base-case gate runs first (notebook-match + idx0-4 solved&verified + a hard idx unsolved),
# then the resumable sweep over the full 1190 (or the shard).
run_greedy.run_sweep(CAP, BUDGET, max_len, data_path=DATA_PATH, out_dir=OUT_DIR,
                     workers=WORKERS, shard=shard, run_base=True)

### One-time helpers
- Run the **labels** block once on any box -> `labels_1190.json` (index-only, no compute).
- Run the **index** block after all 4 streams are collected in `OUT_DIR` -> `solved_path_len_index.json`
  (minimum path length per solved presentation, across streams).

In [ ]:
import json

# Phase 0 labels (run ONCE on any box)
import labels_phase0
labels = labels_phase0.build_labels(DATA_PATH)
with open(os.path.join(OUT_DIR, 'labels_1190.json'), 'w') as f:
    json.dump(labels, f)
print('labels ->', os.path.join(OUT_DIR, 'labels_1190.json'))

# Minimum-path-length index (run AFTER all 4 streams are in OUT_DIR)
import build_index
idx, counts = build_index.build(OUT_DIR)
if idx:
    with open(os.path.join(OUT_DIR, 'solved_path_len_index.json'), 'w') as f:
        json.dump({str(k): idx[k] for k in sorted(idx)}, f, indent=0)
    print(len(idx), 'solved+verified idx; per-stream:', counts)
else:
    print('no streams yet — run the sweep boxes first')